In [ ]:

import os

# Search for the sales dataset inside /var
for root, dirs, files in os.walk("/var"):
    if "sales_pipeline.csv" in files:
        print("Dataset found at:", root)

In [ ]:

import pandas as pd

# Load datasets from the root directory
sales = pd.read_csv("/sales_pipeline.csv")
accounts = pd.read_csv("/accounts.csv")
products = pd.read_csv("/products.csv")
sales_teams = pd.read_csv("/sales_teams.csv")
data_dictionary = pd.read_csv("/data_dictionary.csv")

# Display dataset dimensions
print("Sales Pipeline:", sales.shape)
print("Accounts:", accounts.shape)
print("Products:", products.shape)
print("Sales Teams:", sales_teams.shape)
print("Data Dictionary:", data_dictionary.shape)

print("\nAll datasets loaded successfully!")


In [ ]:

# Display column names for each dataset

datasets = {
    "Sales Pipeline": sales,
    "Accounts": accounts,
    "Products": products,
    "Sales Teams": sales_teams,
    "Data Dictionary": data_dictionary
}

for name, df in datasets.items():
    print(f"\n{name} Columns:")
    print(df.columns.tolist())


In [ ]:

# Step 2.7 - Data Quality Assessment

for name, df in datasets.items():

    print("\n" + "=" * 45)
    print(name.upper())
    print("=" * 45)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)


In [ ]:

# Check Sales Pipeline and Accounts

for name, df in {
    "Sales Pipeline": sales,
    "Accounts": accounts
}.items():

    print("\n" + "=" * 45)
    print(name.upper())
    print("=" * 45)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)


In [ ]:

# Step 2.8 - Missing Values by Deal Stage

columns_to_check = [
    "account",
    "engage_date",
    "close_date",
    "close_value"
]

missing_by_stage = sales.groupby("deal_stage")[
    columns_to_check
].apply(lambda df: df.isnull().sum())

print("Missing Values by Deal Stage:")
print(missing_by_stage)

print("\nDeal Stage Distribution:")
print(sales["deal_stage"].value_counts())


In [ ]:

# Step 2.9 - Convert Date Columns

# Create a working copy to preserve original data
sales_clean = sales.copy()

# Convert text dates to datetime
sales_clean["engage_date"] = pd.to_datetime(
    sales_clean["engage_date"],
    errors="coerce"
)

sales_clean["close_date"] = pd.to_datetime(
    sales_clean["close_date"],
    errors="coerce"
)

# Verify date column types
print("Updated Data Types:")
print(sales_clean[["engage_date", "close_date"]].dtypes)

print("\nMissing Values After Conversion:")
print(sales_clean[["engage_date", "close_date"]].isnull().sum())

print("\nFirst 5 Records:")
print(sales_clean[
    ["opportunity_id", "engage_date", "close_date"]
].head())


In [ ]:

# Step 2.10 - Handle Missing Customer Accounts

# Replace missing account names
sales_clean["account"] = sales_clean["account"].fillna(
    "Unknown Account"
)

# Verify missing values
print("Missing Account Values:")
print(sales_clean["account"].isnull().sum())

print("\nUnknown Account Records:")
print(
    (sales_clean["account"] == "Unknown Account").sum()
)

print("\nTotal Sales Records:")
print(len(sales_clean))


In [ ]:

# Step 2.11 - Handle Missing Parent Company Information

# Create a copy of the original accounts dataset
accounts_clean = accounts.copy()

# Replace missing parent company information
accounts_clean["subsidiary_of"] = (
    accounts_clean["subsidiary_of"].fillna("Not Specified")
)

# Verify the results
print("Missing Parent Company Values:")
print(accounts_clean["subsidiary_of"].isnull().sum())

print("\nNot Specified Records:")
print(
    (accounts_clean["subsidiary_of"] == "Not Specified").sum()
)

print("\nTotal Account Records:")
print(len(accounts_clean))


In [ ]:

# Step 2.12 - Validate Deal Stages and Revenue

print("Deal Stage Distribution:")
print(sales_clean["deal_stage"].value_counts())

print("\nRevenue Statistics:")
print(sales_clean["close_value"].describe())

print("\nNegative Revenue Records:")
print((sales_clean["close_value"] < 0).sum())

print("\nWon Deals with Missing Revenue:")
print(
    sales_clean.loc[
        sales_clean["deal_stage"] == "Won",
        "close_value"
    ].isnull().sum()
)

print("\nOpen Deals with Recorded Revenue:")
print(
    sales_clean.loc[
        sales_clean["deal_stage"].isin(
            ["Prospecting", "Engaging"]
        ),
        "close_value"
    ].notnull().sum()
)


In [ ]:

# Step 2.13 - Revenue Validation by Deal Stage

revenue_validation = sales_clean.groupby("deal_stage").agg(
    total_deals=("opportunity_id", "count"),
    zero_values=("close_value", lambda x: (x == 0).sum()),
    positive_values=("close_value", lambda x: (x > 0).sum()),
    total_close_value=("close_value", "sum")
)

print("Revenue Validation by Deal Stage:")
print(revenue_validation)

print("\nWon Deals with Zero Closing Value:")
print(
    ((sales_clean["deal_stage"] == "Won") &
     (sales_clean["close_value"] == 0)).sum()
)

print("\nLost Deals with Positive Closing Value:")
print(
    ((sales_clean["deal_stage"] == "Lost") &
     (sales_clean["close_value"] > 0)).sum()
)


In [ ]:

# Step 2.14 - Validate Dataset Relationships

print("Duplicate Account Names:")
print(accounts_clean["account"].duplicated().sum())

print("\nDuplicate Product Names:")
print(products["product"].duplicated().sum())

print("\nDuplicate Sales Agent Names:")
print(sales_teams["sales_agent"].duplicated().sum())

print("\nUnmatched Customer Accounts:")
print(
    (~sales_clean.loc[
        sales_clean["account"] != "Unknown Account",
        "account"
    ].isin(accounts_clean["account"])).sum()
)

print("\nUnmatched Products:")
print(
    (~sales_clean["product"].isin(products["product"])).sum()
)

print("\nUnmatched Sales Agents:")
print(
    (~sales_clean["sales_agent"].isin(
        sales_teams["sales_agent"]
    )).sum()
)


In [ ]:

# Step 2.14.1 - Investigate Unmatched Products

unmatched_products = sales_clean.loc[
    ~sales_clean["product"].isin(products["product"]),
    "product"
]

print("Unmatched Product Names:")
print(unmatched_products.value_counts())

print("\nValid Product Names:")
print(products["product"].tolist())

print("\nTotal Unmatched Records:")
print(len(unmatched_products))


In [ ]:

# Step 2.14.2 - Standardize Product Names

# Correct the inconsistent product name
sales_clean["product"] = sales_clean["product"].replace({
    "GTXPro": "GTX Pro"
})

# Verify unmatched products
unmatched_count = (
    ~sales_clean["product"].isin(products["product"])
).sum()

print("Unmatched Products After Cleaning:")
print(unmatched_count)

print("\nGTX Pro Records:")
print(
    (sales_clean["product"] == "GTX Pro").sum()
)

print("\nTotal Sales Records:")
print(len(sales_clean))


In [ ]:

# Step 2.15 - Merge Sales Datasets

# Start with cleaned sales pipeline
master_sales = sales_clean.copy()

# Merge customer information
master_sales = master_sales.merge(
    accounts_clean,
    on="account",
    how="left",
    validate="many_to_one"
)

# Merge product information
master_sales = master_sales.merge(
    products,
    on="product",
    how="left",
    validate="many_to_one"
)

# Merge sales team information
master_sales = master_sales.merge(
    sales_teams,
    on="sales_agent",
    how="left",
    validate="many_to_one"
)

# Verify merged dataset
print("Master Dataset Shape:")
print(master_sales.shape)

print("\nTotal Records:")
print(len(master_sales))

print("\nMaster Dataset Columns:")
print(master_sales.columns.tolist())

print("\nMissing Product Prices:")
print(master_sales["sales_price"].isnull().sum())

print("\nMissing Sales Managers:")
print(master_sales["manager"].isnull().sum())


In [ ]:

# Step 2.16 - Final Master Dataset Quality Check

print("MASTER SALES DATA QUALITY REPORT")
print("=" * 45)

# Check dataset dimensions
print("\nDataset Shape:")
print(master_sales.shape)

# Check unique opportunity IDs
print("\nDuplicate Opportunity IDs:")
print(master_sales["opportunity_id"].duplicated().sum())

# Check missing values
print("\nMissing Values:")
print(master_sales.isnull().sum())

# Verify won revenue
won_revenue = master_sales.loc[
    master_sales["deal_stage"] == "Won",
    "close_value"
].sum()

print("\nTotal Won Revenue:")
print(won_revenue)

# Check deal stage distribution
print("\nDeal Stage Distribution:")
print(master_sales["deal_stage"].value_counts())


In [ ]:

# Step 2.17 - Export Cleaned Master Dataset

from google.colab import files

# Export the master dataset to CSV
master_sales.to_csv(
    "master_sales_cleaned.csv",
    index=False
)

print("Master sales dataset exported successfully!")

print("Total Records:", len(master_sales))
print("Total Columns:", len(master_sales.columns))

# Download CSV to your computer
files.download("master_sales_cleaned.csv")
